# Helmet & reflective-jacket detection (YOLO11)

This notebook trains the same model used by **Supervisor Tier 1** verification:

- After training, weights are saved under `ppe_training/yolov11_ppe/weights/best.pt` inside this repo (paths are pinned to the dataset’s parent folder so runs do not go to `Downloads/.../runs` when the notebook cwd differs).
- The Flask app `ppe_server.py` loads that path and exposes `POST /verify_helmet` for the React Supervisor UI (`Tier1.tsx`).

**Classes** (see `safety-Helmet-Reflective-Jacket/data.yaml`): `0` = Safety-Helmet, `1` = Reflective-Jacket.

**Environment:** use the project venv so `ultralytics` matches training:

```bash
cd /home/khushiii/vikaas
source training_env/bin/activate   # or: ./training_env/bin/python -m pip install jupyter ipykernel
python -m ipykernel install --user --name=vikaas-ppe --display-name="Python (vikaas PPE)"
```

Then select kernel **Python (vikaas PPE)** in Jupyter.

In [3]:
import os
from pathlib import Path

# Notebook cwd is often the notebook folder — anchor to repo root
ROOT = Path.cwd()
if (ROOT / "safety-Helmet-Reflective-Jacket" / "data.yaml").exists():
    pass
elif (ROOT.parent / "safety-Helmet-Reflective-Jacket" / "data.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Working directory:", os.getcwd())

DATA_YAML = ROOT / "safety-Helmet-Reflective-Jacket" / "data.yaml"
assert DATA_YAML.is_file(), f"Missing {DATA_YAML}"
# Same root as train_ppe_v2.py / ppe_server.py (avoids Ultralytics writing under another cwd)
REPO_ROOT = DATA_YAML.parent.parent

Working directory: /home/khushiii/vikaas


In [4]:
%pip install -q pyyaml ultralytics

Note: you may need to restart the kernel to use updated packages.


In [5]:
import yaml  # pyright: ignore[reportMissingModuleSource]

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)
print("Dataset config:", data_cfg)

base = Path(data_cfg.get("path", DATA_YAML.parent))
train_img = (base / data_cfg["train"]).resolve()
n_train = len([p for p in train_img.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}])
print(f"Training images counted: {n_train} under {train_img}")

Dataset config: {'path': '/home/khushiii/vikaas/safety-Helmet-Reflective-Jacket', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'names': {0: 'Safety-Helmet', 1: 'Reflective-Jacket'}, 'nc': 2}
Training images counted: 7350 under /home/khushiii/vikaas/safety-Helmet-Reflective-Jacket/train/images


In [ ]:
import os
import shutil
from pathlib import Path
import torch
import yaml  # pyright: ignore[reportMissingModuleSource]
from ultralytics import YOLO

# After kernel restart / %pip, REPO_ROOT may be missing — re-resolve from data.yaml
if "REPO_ROOT" not in globals():
    _here = Path.cwd()
    for _c in (_here, _here.parent):
        _dy = _c / "safety-Helmet-Reflective-Jacket" / "data.yaml"
        if _dy.is_file():
            os.chdir(_c)
            ROOT = _c
            DATA_YAML = _dy
            REPO_ROOT = DATA_YAML.parent.parent
            break
    else:
        raise FileNotFoundError(
            "Could not find safety-Helmet-Reflective-Jacket/data.yaml — open the notebook from the vikaas repo and run the first setup cell."
        )

if "DATA_YAML" not in globals():
    DATA_YAML = REPO_ROOT / "safety-Helmet-Reflective-Jacket" / "data.yaml"
if "ROOT" not in globals():
    ROOT = REPO_ROOT

# Fast run: first N train images (sorted by filename) + 10 epochs + smaller imgsz.
# Full run: all train images + 50 epochs. If you switch modes, delete `ppe_training/yolov11_ppe` once.
FAST_TRAIN = True
FAST_TRAIN_N = 700  # first N images from train/images (after sorting filenames)
EPOCHS = 10 if FAST_TRAIN else 50
IMGSZ = 416 if FAST_TRAIN else 640

# Pretrained weights in repo root: yolo11n.pt (default) or yolo26n.pt
PRETRAINED = REPO_ROOT / "yolo11n.pt"  # change to yolo26n.pt if you prefer YOLO26 nano

_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _build_first_n_train_yaml(repo_root: Path, data_yaml: Path, n: int) -> Path:
    """Symlink first N train images + labels into ppe_training/subset_train_<n> and write a data yaml."""
    with open(data_yaml) as f:
        cfg = yaml.safe_load(f)
    base = Path(cfg.get("path", data_yaml.parent))
    train_img_dir = (base / cfg["train"]).resolve()
    labels_dir = train_img_dir.parent / "labels"
    imgs = sorted(
        [p for p in train_img_dir.glob("*") if p.suffix.lower() in _EXT],
        key=lambda p: p.name,
    )
    take = imgs[:n]
    if len(take) < n:
        raise ValueError(f"Only {len(take)} train images found; need at least {n}")

    sub_root = repo_root / "ppe_training" / f"subset_train_{n}"
    sub_img = sub_root / "images"
    sub_lbl = sub_root / "labels"
    if sub_root.exists():
        shutil.rmtree(sub_root)
    sub_img.mkdir(parents=True)
    sub_lbl.mkdir(parents=True)

    for img_path in take:
        stem = img_path.stem
        (sub_img / img_path.name).symlink_to(img_path.resolve())
        src_l = labels_dir / f"{stem}.txt"
        if not src_l.is_file():
            raise FileNotFoundError(f"Missing label for {img_path.name}: {src_l}")
        (sub_lbl / f"{stem}.txt").symlink_to(src_l.resolve())

    out = repo_root / "ppe_training" / f"data_subset_train_{n}.yaml"
    rel_ds = "safety-Helmet-Reflective-Jacket"
    yml = {
        "path": str(repo_root),
        "train": f"ppe_training/subset_train_{n}/images",
        "val": f"{rel_ds}/valid/images",
        "test": f"{rel_ds}/test/images",
        "names": cfg.get("names", {0: "Safety-Helmet", 1: "Reflective-Jacket"}),
        "nc": int(cfg.get("nc", 2)),
    }
    with open(out, "w") as f:
        yaml.safe_dump(yml, f, sort_keys=False, allow_unicode=True)
    print(f"Subset: {len(take)} images (first {n} by filename) → {sub_root}")
    return out


if FAST_TRAIN:
    train_data_yaml = _build_first_n_train_yaml(REPO_ROOT, DATA_YAML, FAST_TRAIN_N)
else:
    train_data_yaml = DATA_YAML

device = "0" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print(f"FAST_TRAIN={FAST_TRAIN} | train_yaml={train_data_yaml.name} epochs={EPOCHS} imgsz={IMGSZ}")

run_weights = REPO_ROOT / "ppe_training" / "yolov11_ppe" / "weights"
last_pt = run_weights / "last.pt"
resume = last_pt.is_file()
if resume:
    print("Resuming from interrupted run:", last_pt)
    model = YOLO(str(last_pt))
else:
    assert PRETRAINED.is_file(), f"Missing {PRETRAINED}"
    model = YOLO(str(PRETRAINED))

results = model.train(
    data=str(train_data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    patience=0,
    device=device,
    project=str(REPO_ROOT / "ppe_training"),
    name="yolov11_ppe",
    exist_ok=True,
    resume=resume,
)
_PPE_IMGSZ = IMGSZ  # val / inference cells use this when set
results


Device: cpu
FAST_TRAIN=True | fraction=0.1 epochs=10 imgsz=416
Resuming from interrupted run: /home/khushiii/vikaas/ppe_training/yolov11_ppe/weights/last.pt
Ultralytics 8.4.24 🚀 Python-3.14.3 torch-2.10.0+cu128 CPU (12th Gen Intel Core i5-1235U)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/khushiii/vikaas/safety-Helmet-Reflective-Jacket/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0,

## Validation metrics (Supervisor model QA)

Runs mAP on the `val` split from `data.yaml` using the best checkpoint.

In [ ]:
import os
from pathlib import Path
import torch
from ultralytics import YOLO

if "REPO_ROOT" not in globals():
    _here = Path.cwd()
    for _c in (_here, _here.parent):
        _dy = _c / "safety-Helmet-Reflective-Jacket" / "data.yaml"
        if _dy.is_file():
            os.chdir(_c)
            ROOT = _c
            DATA_YAML = _dy
            REPO_ROOT = DATA_YAML.parent.parent
            break
    else:
        raise FileNotFoundError("Run the setup cell first or re-run the training cell’s path block.")

best_pt = REPO_ROOT / "ppe_training" / "yolov11_ppe" / "weights" / "best.pt"
assert best_pt.is_file(), f"Train first or check path: {best_pt}"

_dev = "0" if torch.cuda.is_available() else "cpu"
_imgsz = globals().get("_PPE_IMGSZ", 640)
m = YOLO(str(best_pt))
metrics = m.val(data=str(DATA_YAML), imgsz=_imgsz, device=_dev)
print(metrics)

In [ ]:
# Optional: run inference on one validation image (same classes as /verify_helmet)
import os
from pathlib import Path
import torch
from ultralytics import YOLO

if "REPO_ROOT" not in globals():
    _here = Path.cwd()
    for _c in (_here, _here.parent):
        _dy = _c / "safety-Helmet-Reflective-Jacket" / "data.yaml"
        if _dy.is_file():
            os.chdir(_c)
            ROOT = _c
            DATA_YAML = _dy
            REPO_ROOT = DATA_YAML.parent.parent
            break
    else:
        raise FileNotFoundError("Run the setup cell first or re-run the training cell’s path block.")

best_pt = REPO_ROOT / "ppe_training" / "yolov11_ppe" / "weights" / "best.pt"
val_dir = ROOT / "safety-Helmet-Reflective-Jacket" / "valid" / "images"
sample = next(val_dir.glob("*.jpg"), None) or next(val_dir.glob("*.png"), None)
if sample and best_pt.is_file():
    _dev = "0" if torch.cuda.is_available() else "cpu"
    m = YOLO(str(best_pt))
    _isz = globals().get("_PPE_IMGSZ", 640)
    r = m.predict(sample, conf=0.5, verbose=False, device=_dev, imgsz=_isz)[0]
    for b in r.boxes:
        print(int(b.cls), r.names[int(b.cls)], float(b.conf))
else:
    print("Need trained best.pt and a val image", best_pt, val_dir)